# E-Commerce Product Demand Prediction
### B104 Artificial Intelligence & Machine Learning
### Sahil Pathania GH1041137

### GitHub Link :- https://github.com/sahilpathania06/E-Commerce-Demand-Prediction-.git

# Business Problem and Machine Learning Problem

This regression project uses historical sales to predict daily product demand for inventory planning, reducing overstock and stockout risks.

# Data Understanding

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Sales Dataset

The sales dataset contains historical transactions. Since the original dataset is very large, the first 100,000 records were used for initial analysis.

In [2]:
sales = pd.read_csv("../Data/raw/Sales.csv", nrows = 100000)

sales.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,date_,city_name,order_id,cart_id,dim_customer_key,procured_quantity,unit_selling_price,total_discount_amount,product_id,total_weighted_landing_price
0,0,0,0,2022-04-01,Mumbai,112246974,173273802,17995199,1,234.0,0.0,344107,202.513030
1,1,1,1,2022-04-01,Bengaluru,112246976,173273597,18259433,1,64.0,0.0,389676,48.714375
2,2,2,2,2022-04-01,Bengaluru,112247019,173123717,5402601,1,1031.0,0.0,39411,975.996000
3,3,3,3,2022-04-01,HR-NCR,112247045,172547459,15649744,1,57.0,0.0,369742,25.000000
4,4,4,4,2022-04-01,Mumbai,112247123,173081820,10127605,2,30.0,0.0,12872,57.980004


### Descriptive Statistics

In [3]:
sales.describe()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,order_id,cart_id,dim_customer_key,procured_quantity,unit_selling_price,total_discount_amount,product_id,total_weighted_landing_price
count,100000.000000,100000.000000,100000.000000,1.000000e+05,1.000000e+05,1.000000e+05,100000.000000,100000.00000,100000.000000,100000.000000,99992.000000
mean,49999.500000,49999.500000,49999.500000,1.131003e+08,1.735833e+08,9.795988e+06,1.317210,89.30556,0.434280,222612.078750,92.489938
std,28867.657797,28867.657797,28867.657797,6.349947e+05,5.720867e+06,6.138119e+06,0.997175,114.53344,7.486231,184642.048137,132.790259
min,0.000000,0.000000,0.000000,1.122394e+08,1.212636e+08,2.570000e+02,0.000000,0.00000,0.000000,1.000000,0.015683
25%,24999.750000,24999.750000,24999.750000,1.125204e+08,1.733209e+08,3.851152e+06,1.000000,26.00000,0.000000,19507.000000,24.870000
50%,49999.500000,49999.500000,49999.500000,1.129971e+08,1.744303e+08,1.042260e+07,1.000000,50.00000,0.000000,217614.000000,48.972000
75%,74999.250000,74999.250000,74999.250000,1.136533e+08,1.764462e+08,1.577367e+07,1.000000,105.00000,0.000000,397484.000000,105.598010
max,99999.000000,99999.000000,99999.000000,1.145197e+08,1.794337e+08,1.845251e+07,50.000000,5450.00000,1000.000000,482283.000000,5027.086400


The descriptive statistics summarise the numerical features in Flipkart’s sales dataset. The “Unnamed” columns will be removed during preprocessing.

## Products Dataset

In [4]:
products = pd.read_csv("../Data/raw/products.csv")

products.head()

,Unnamed: 0,product_id,product_name,unit,product_type,brand_name,manufacturer_name,l0_category,l1_category,l2_category,l0_category_id,l1_category_id,l2_category_id
0,0,476763,Christmas - Card,1 unit,Card,NaN,HOT,Specials,Bill Breaker,Bill Breaker,343,1741,1741
1,1,483436,Plum BodyLovin' Hawaiian Rumba Shower Gel - Sa...,20 ml,Sample,Plum BodyLovin',Pureplay Skin Sciences India Pvt. Ltd.,Specials,Free Store,Free Store,343,1493,1493
2,2,476825,Diwali Gift Card Free - Sample,1 unit,Sample,NaN,HOT,Specials,Bill Breaker,Bill Breaker,343,1741,1741
3,3,483438,Plum BodyLovin' Trippin' Mimosas Shower Gel - ...,20 ml,Sample,Plum BodyLovin',Pureplay Skin Sciences India Pvt. Ltd.,Specials,Free Store,Free Store,343,1493,1493
4,4,480473,Flipkart Valentine Day Greeting - Card,1 unit,Card,Flipkart,Dummy Manufacturer,Specials,Bill Breaker,Bill Breaker,343,1741,1741


The first five rows of this dataset shows descriptive information about each product including name, type and brand. The dataset has 13 columns.

### Descriptive Statistics

In [5]:
products.describe()

,Unnamed: 0,product_id,l0_category_id,l1_category_id,l2_category_id
count,32226.000000,32226.000000,32226.000000,32226.000000,32226.000000
mean,16112.500000,401505.635263,602.077856,912.249364,819.395488
std,9302.989224,147362.043573,627.878785,484.406205,607.573456
min,0.000000,1.000000,4.000000,19.000000,6.000000
25%,8056.250000,396231.500000,15.000000,707.000000,133.000000
50%,16112.500000,477867.500000,175.000000,975.000000,958.000000
75%,24168.750000,489523.750000,1379.000000,1160.000000,1177.000000
max,32225.000000,498814.000000,1557.000000,2039.000000,2039.000000


## Relationship between Sales and Products Datasets

The sales dataset contains transaction-level data, while the products dataset provides details about individual products. Both include product_id, allowing the datasets to be linked.

# Data Preprocessing

## Removing Unwanted Columns

In [6]:
sales = sales.drop(columns=["Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0"])
products = products.drop(columns=["Unnamed: 0"])

## Missing Values

In [7]:
print("Sales Missing Values : ")
print(sales.isnull().sum())

print("\n\nProducts Missing Values : ")
print(products.isnull().sum())

Sales Missing Values : 
date_                           0
city_name                       0
order_id                        0
cart_id                         0
dim_customer_key                0
procured_quantity               0
unit_selling_price              0
total_discount_amount           0
product_id                      0
total_weighted_landing_price    8
dtype: int64


Products Missing Values : 
product_id              0
product_name            0
unit                    0
product_type            0
brand_name           1438
manufacturer_name    2416
l0_category             0
l1_category             0
l2_category             0
l0_category_id          0
l1_category_id          0
l2_category_id          0
dtype: int64


### Handling Missing Values

Missing manufacturer and brand values were replaced with “Unknown” instead of removing the products, preserving the other data needed for analysis.

In [8]:
products["brand_name"] = products["brand_name"].fillna("Unknown")
products["manufacturer_name"] = products["manufacturer_name"].fillna("Unknown")

sales[sales["total_weighted_landing_price"].isnull()]

,date_,city_name,order_id,cart_id,dim_customer_key,procured_quantity,unit_selling_price,total_discount_amount,product_id,total_weighted_landing_price
19779,2022-04-02,Mumbai,112656306,171443518,13999538,2,15.0,0.0,481193,NaN
30613,2022-04-06,Mumbai,113467653,173769335,1365002,0,0.0,0.0,478908,NaN
47783,2022-04-06,Bengaluru,113467306,172018623,17917824,1,0.0,0.0,478908,NaN
47790,2022-04-06,HR-NCR,113467520,175428058,1181803,1,0.0,0.0,478908,NaN
51710,2022-04-06,Bengaluru,113468423,176629143,18129608,1,0.0,0.0,478908,NaN
52446,2022-04-06,Mumbai,113467855,176209026,2472921,1,0.0,0.0,478908,NaN
52454,2022-04-06,Mumbai,113468128,176628464,17854513,0,0.0,0.0,478908,NaN
52459,2022-04-06,HR-NCR,113468471,176628466,17928466,1,0.0,0.0,478908,NaN


Eight rows with missing total_weighted_landing_price were removed from the initial 100,000-row sample only. Full-data aggregation rereads the original CSV and does not use price.

In [9]:
sales = sales.dropna(subset=["total_weighted_landing_price"])

In [10]:
print("Sales Missing Values : ")
print(sales.isnull().sum())

print("\n Products Missing Value : ")
print(products.isnull().sum())

Sales Missing Values : 
date_                           0
city_name                       0
order_id                        0
cart_id                         0
dim_customer_key                0
procured_quantity               0
unit_selling_price              0
total_discount_amount           0
product_id                      0
total_weighted_landing_price    0
dtype: int64

 Products Missing Value : 
product_id           0
product_name         0
unit                 0
product_type         0
brand_name           0
manufacturer_name    0
l0_category          0
l1_category          0
l2_category          0
l0_category_id       0
l1_category_id       0
l2_category_id       0
dtype: int64


## Duplicate Records

In [11]:
print("Duplicate Sale Rows : ", sales.duplicated().sum())
print("Duplicate Product Rows : ", products.duplicated().sum())

Duplicate Sale Rows :  0
Duplicate Product Rows :  0


No duplicates were found in both datasets.

## Data Types

In [12]:
print("Sales Data Type : ")
print(sales.dtypes)

print("Product Data Type : ")
print(products.dtypes)

Sales Data Type : 
date_                               str
city_name                           str
order_id                          int64
cart_id                           int64
dim_customer_key                  int64
procured_quantity                 int64
unit_selling_price              float64
total_discount_amount           float64
product_id                        int64
total_weighted_landing_price    float64
dtype: object
Product Data Type : 
product_id           int64
product_name           str
unit                   str
product_type           str
brand_name             str
manufacturer_name      str
l0_category            str
l1_category            str
l2_category            str
l0_category_id       int64
l1_category_id       int64
l2_category_id       int64
dtype: object


Most of the columns are stored in right data types. However, the date is stored as string. We are converting date into datetime format.

In [13]:
sales["date_"] = pd.to_datetime(sales["date_"])
print(sales["date_"].dtype)

datetime64[us]


## Invalid Values

Important numerical variables were checked for invalid values. Quantities and prices, for example, should not be negative, as this could distort the analysis and model training.

In [14]:
print("Negative Quantity:", (sales["procured_quantity"] < 0).sum())
print("Negative Selling Price:", (sales["unit_selling_price"] < 0).sum())
print("Negative Discount:", (sales["total_discount_amount"] < 0).sum())
print("Negative Landing Price:", (sales["total_weighted_landing_price"] < 0).sum())

print("\nZero Quantity:", (sales["procured_quantity"] == 0).sum())
print("Zero Selling Price:", (sales["unit_selling_price"] == 0).sum())

Negative Quantity: 0
Negative Selling Price: 0
Negative Discount: 0
Negative Landing Price: 0

Zero Quantity: 611
Zero Selling Price: 0


No negative values were found in the checked numerical columns of the initial 100,000-row Sales sample.

## Preparing the Full Sales Dataset

The initial sample helped identify the necessary preprocessing steps. Because the complete Sales dataset is very large, it was processed in smaller batches to reduce memory usage. The full date range was checked before creating the demand dataset.

In [15]:
min_date = None
max_date = None

for chunk in pd.read_csv("../Data/raw/Sales.csv", usecols=["date_"], chunksize=500000):
    chunk["date_"] = pd.to_datetime(chunk["date_"])
    
    chunk_min = chunk["date_"].min()
    chunk_max = chunk["date_"].max()
    
    if min_date is None or chunk_min < min_date:
        min_date = chunk_min
        
    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

print("First Date:", min_date)
print("Last Date:", max_date)

First Date: 2022-04-01 00:00:00
Last Date: 2022-07-10 00:00:00


The complete Sales dataset covers 1 April to 10 July 2022. Summed procured_quantity by date, city and product is used as a proxy for demand; it does not measure unmet demand. Only recorded combinations are included, without filling absent combinations with zeros.

### Creating Daily Product Demand

In [16]:
demand_chunks = []

for chunk in pd.read_csv(
    "../Data/raw/Sales.csv",
    usecols=["date_", "city_name", "product_id", "procured_quantity"],
    chunksize=500000
):
    chunk["date_"] = pd.to_datetime(chunk["date_"])

    chunk_demand = chunk.groupby(
        ["date_", "city_name", "product_id"],
        as_index=False
    )["procured_quantity"].sum()

    demand_chunks.append(chunk_demand)

daily_demand = pd.concat(demand_chunks)

daily_demand = daily_demand.groupby(
    ["date_", "city_name", "product_id"],
    as_index=False
)["procured_quantity"].sum()

daily_demand = daily_demand.rename(
    columns={"procured_quantity": "demand"}
)

daily_demand.head()

,date_,city_name,product_id,demand
0,2022-04-01,Bengaluru,1,61
1,2022-04-01,Bengaluru,2,36
2,2022-04-01,Bengaluru,3,7
3,2022-04-01,Bengaluru,4,32
4,2022-04-01,Bengaluru,7,132


### Aggregated Dataset Size

In [17]:
daily_demand.shape

(1764981, 4)

In [18]:
daily_demand["demand"].describe()

count    1.764981e+06
mean     3.409447e+01
std      1.701989e+02
min      0.000000e+00
25%      2.000000e+00
50%      6.000000e+00
75%      1.900000e+01
max      1.032300e+04
Name: demand, dtype: float64

The aggregated dataset contains 1,764,981 rows and 4 columns.

### Merging Product Information

In [19]:
final_data = daily_demand.merge(
    products,
    on="product_id",
    how="left"
)

final_data.head()

,date_,city_name,product_id,demand,product_name,unit,product_type,brand_name,manufacturer_name,l0_category,l1_category,l2_category,l0_category_id,l1_category_id,l2_category_id
0,2022-04-01,Bengaluru,1,61,Nutrela Soya Mini Chunks,200 g,Soya Mini Chunks,Nutrela,RUCHI SOYA INDUSTRIES,"Atta, Rice & Dal","Rajma, Chhole & Others",Soya,16.0,1573.0,1297.0
1,2022-04-01,Bengaluru,2,36,Nutrela Soya Chunks,220 g,Soya Chunks,Nutrela,RUCHI SOYA INDUSTRIES,"Atta, Rice & Dal","Rajma, Chhole & Others",Soya,16.0,1573.0,1297.0
2,2022-04-01,Bengaluru,3,7,Nutrela Soya Granules,220 g,Soya Granules,Nutrela,RUCHI SOYA INDUSTRIES,"Atta, Rice & Dal","Rajma, Chhole & Others",Soya,16.0,1573.0,1297.0
3,2022-04-01,Bengaluru,4,32,Fortune Soya Chunks,200 g,Soya Chunks,Fortune,ADANI WILMAR LIMITED,"Atta, Rice & Dal","Rajma, Chhole & Others",Soya,16.0,1573.0,1297.0
4,2022-04-01,Bengaluru,7,132,Aashirvaad Select Whole Wheat Sharbati Atta,5 kg,Sharbati Atta,Aashirvaad Select,ITC Limited,"Atta, Rice & Dal",Atta,Atta,16.0,1165.0,1165.0


In [20]:
print("Dataset Shape:", final_data.shape)
print("Missing Product Names:", final_data["product_name"].isnull().sum())

Dataset Shape: (1764981, 15)
Missing Product Names: 40913


The joint dataset contains 1,764,981 records and 15 columns. However, 40,913 records lack product information, suggesting that some product_id values in the Sales dataset are missing from the Products dataset.

In [21]:
unmatched = final_data[final_data["product_name"].isnull()]

print("Unmatched Rows:", len(unmatched))
print("Unique Unmatched Products:", unmatched["product_id"].nunique())

unmatched[["product_id", "demand"]].head(10)

Unmatched Rows: 40913
Unique Unmatched Products: 1496


,product_id,demand
273,10624,8
414,13971,7
582,19734,28
672,23827,9
982,45744,2
1034,56748,2
1383,118576,2
1567,200953,0
1570,201583,1
1874,333420,16


### Handling Unmatched Product Information

The unmatched products are retained because their demand information is still useful for analysis. Missing descriptive product attributes are replaced with "Unknown".

In [22]:
text_columns = [
    "product_name",
    "unit",
    "product_type",
    "brand_name",
    "manufacturer_name",
    "l0_category",
    "l1_category",
    "l2_category"
]

final_data[text_columns] = final_data[text_columns].fillna("Unknown")

In [23]:
id_columns = [
    "l0_category_id",
    "l1_category_id",
    "l2_category_id"
]

final_data[id_columns] = final_data[id_columns].fillna(-1)

In [24]:
print("Missing Values:", final_data.isnull().sum().sum())
print("Duplicate Rows:", final_data.duplicated().sum())
print("Dataset Shape:", final_data.shape)

Missing Values: 0
Duplicate Rows: 0
Dataset Shape: (1764981, 15)


The final processed dataset contains 1,764,981 observations and 15 columns. No missing values or duplicate records remain after preprocessing.

### Saving the Processed Dataset

The cleaned and aggregated dataset is saved separately.

In [25]:
final_data.to_csv("../Data/cleaned/demand.csv", index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


# Feature Engineering

Date-based features help the models identify temporal demand patterns.

In [26]:
final_data["date_"] = pd.to_datetime(final_data["date_"])

final_data["day"] = final_data["date_"].dt.day
final_data["month"] = final_data["date_"].dt.month
final_data["day_of_week"] = final_data["date_"].dt.dayofweek

final_data[["date_", "day", "month", "day_of_week"]].head()

,date_,day,month,day_of_week
0,2022-04-01,1,4,4
1,2022-04-01,1,4,4
2,2022-04-01,1,4,4
3,2022-04-01,1,4,4
4,2022-04-01,1,4,4


The date variable was converted into day, month and day of week features.

### Weekend Feature

A weekend feature is created from the day of week.

In [27]:
final_data["weekend"] = (final_data["day_of_week"] >= 5).astype(int)

final_data[["date_", "day_of_week", "weekend"]].head(10)

,date_,day_of_week,weekend
0,2022-04-01,4,0
1,2022-04-01,4,0
2,2022-04-01,4,0
3,2022-04-01,4,0
4,2022-04-01,4,0
5,2022-04-01,4,0
6,2022-04-01,4,0
7,2022-04-01,4,0
8,2022-04-01,4,0
9,2022-04-01,4,0


## Feature Selection

Only relevant variables were selected for model training. Product category, city, and time-related features were included because they may influence demand. Product, brand, and manufacturer names were excluded to avoid unnecessary complexity from their many text values.

In [28]:
features = [
    "l0_category",
    "city_name",
    "day",
    "month",
    "day_of_week",
    "weekend"
]

X = final_data[features]
y = final_data["demand"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

X.head()

Features Shape: (1764981, 6)
Target Shape: (1764981,)


,l0_category,city_name,day,month,day_of_week,weekend
0,"Atta, Rice & Dal",Bengaluru,1,4,4,0
1,"Atta, Rice & Dal",Bengaluru,1,4,4,0
2,"Atta, Rice & Dal",Bengaluru,1,4,4,0
3,"Atta, Rice & Dal",Bengaluru,1,4,4,0
4,"Atta, Rice & Dal",Bengaluru,1,4,4,0


## Train Test Split

Earlier dates are used to train the model, while later dates are used to test it. This follows the project's aim of predicting future demand.

In [29]:
dates = final_data["date_"].drop_duplicates().sort_values()

split_date = dates.iloc[int(len(dates) * 0.8)]

train_rows = final_data["date_"] < split_date
test_rows = final_data["date_"] >= split_date

X_train = X.loc[train_rows]
X_test = X.loc[test_rows]

y_train = y.loc[train_rows]
y_test = y.loc[test_rows]

print("Test period begins:", split_date)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Test period begins: 2022-06-14 00:00:00
Training shape: (1361888, 6)
Testing shape: (403093, 6)


## Encoding Categorical Features

City and product category are converted into numerical columns using one-hot encoding. The encoder learns the categories from the training data and applies the same conversion to the test data.

In [30]:
categorical_columns = ["city_name", "l0_category"]

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

train_encoded = encoder.fit_transform(
    X_train[categorical_columns]
)

test_encoded = encoder.transform(
    X_test[categorical_columns]
)

print("Encoded training shape:", train_encoded.shape)
print("Encoded testing shape:", test_encoded.shape)

Encoded training shape: (1361888, 26)
Encoded testing shape: (403093, 26)


In [31]:
numerical_columns = ["day", "month", "day_of_week", "weekend"]

encoded_names = encoder.get_feature_names_out(categorical_columns)

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_names,
    index=X_train.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_names,
    index=X_test.index
)

X_train_ready = pd.concat(
    [X_train[numerical_columns], train_encoded_df],
    axis=1
)

X_test_ready = pd.concat(
    [X_test[numerical_columns], test_encoded_df],
    axis=1
)

print("Final training shape:", X_train_ready.shape)
print("Final testing shape:", X_test_ready.shape)

Final training shape: (1361888, 30)
Final testing shape: (403093, 30)


## Validation Set

The training data is split into an earlier period for model training and a later period for validation. Validation results help compare models before final testing. This separates model selection from final evaluation (G?ron, 2022).

In [32]:
training_dates = final_data.loc[X_train.index, "date_"]
dates = training_dates.drop_duplicates().sort_values()

validation_date = dates.iloc[int(len(dates) * 0.8)]

earlier_rows = training_dates < validation_date
later_rows = training_dates >= validation_date

X_fit = X_train.loc[earlier_rows]
X_val = X_train.loc[later_rows]

y_fit = y_train.loc[earlier_rows]
y_val = y_train.loc[later_rows]

print("Fitting data:", X_fit.shape)
print("Validation data:", X_val.shape)

Fitting data: (1050979, 6)
Validation data: (310909, 6)


In [33]:
validation_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

fit_encoded = validation_encoder.fit_transform(
    X_fit[categorical_columns]
)

val_encoded = validation_encoder.transform(
    X_val[categorical_columns]
)

X_fit_ready = np.column_stack(
    [X_fit[numerical_columns], fit_encoded]
)

X_val_ready = np.column_stack(
    [X_val[numerical_columns], val_encoded]
)

In [34]:
linear_model = LinearRegression()
linear_model.fit(X_fit_ready, y_fit)

val_predictions = linear_model.predict(X_val_ready)

print("Validation MAE:",
      mean_absolute_error(y_val, val_predictions))

print("Validation RMSE:",
      np.sqrt(mean_squared_error(y_val, val_predictions)))

print("Validation R²:",
      r2_score(y_val, val_predictions))

Validation MAE: 42.05643875403462
Validation RMSE: 163.55876718354295
Validation R²: 0.09277813215282038


Linear Regression achieved a validation MAE of 42.06, RMSE of 163.56, and R^2 of 0.093, indicating limited accuracy. The higher RMSE suggests some predictions contain large errors.

## Feature Scaling for KNN

KNN identifies similar observations using distance. Features are standardised to prevent scale differences from affecting results. The scaler is fitted on training data and applied to validation data. Models and preprocessing use scikit-learn (Pedregosa et al., 2011).

In [35]:
scaler = StandardScaler()

X_fit_scaled = scaler.fit_transform(X_fit_ready)
X_val_scaled = scaler.transform(X_val_ready)

print("Scaled fitting data:", X_fit_scaled.shape)
print("Scaled validation data:", X_val_scaled.shape)

Scaled fitting data: (1050979, 30)
Scaled validation data: (310909, 30)


## KNN Regression

KNN predicts demand by averaging the demand of five similar
observations. A random sample of 20,000 fitting rows is used
to reduce computation time. Validation uses 5,000 sampled rows. Linear Regression uses all fitting rows, so this compares the chosen training setups with different training sizes.

In [36]:
X_knn = pd.DataFrame(X_fit_scaled).sample(
    n=20000, random_state=42
)

y_knn = y_fit.iloc[X_knn.index]

knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_knn.to_numpy(), y_knn)

print("KNN training completed.")

KNN training completed.


In [37]:
X_knn_val = pd.DataFrame(X_val_scaled).sample(
    n=5000, random_state=42
)

y_knn_val = y_val.iloc[X_knn_val.index]

knn_predictions = knn_model.predict(X_knn_val.to_numpy())

print("KNN validation MAE:",
      mean_absolute_error(y_knn_val, knn_predictions))

print("KNN validation RMSE:",
      np.sqrt(mean_squared_error(y_knn_val, knn_predictions)))

print("KNN validation R²:",
      r2_score(y_knn_val, knn_predictions))

KNN validation MAE: 44.04524
KNN validation RMSE: 178.1237599198939
KNN validation R²: -0.07720182747272797


### KNN Validation Results

On the 5,000-row validation sample, KNN achieved MAE 44.05, RMSE 178.12 and R^2 -0.077 using five neighbours, showing weak performance.

## KNN Hyperparameter Tuning

The number of neighbours is a key KNN setting. Different n_neighbors values were tested to find the lowest validation error. A training sample was used because KNN is computationally demanding with large datasets.

In [38]:
k_values = [3, 5, 7, 9, 11]

knn_results = []

for k in k_values:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_knn.to_numpy(), y_knn)

    predictions = model.predict(X_knn_val.to_numpy())

    mae = mean_absolute_error(y_knn_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_knn_val, predictions))
    r2 = r2_score(y_knn_val, predictions)

    knn_results.append([k, mae, rmse, r2])

knn_results = pd.DataFrame(
    knn_results,
    columns=["K", "MAE", "RMSE", "R2"]
)

knn_results

,K,MAE,RMSE,R2
0,3,45.614067,188.664581,-0.208465
1,5,44.045240,178.123760,-0.077202
2,7,43.557057,171.698680,-0.000892
3,9,42.776089,167.792858,0.044127
4,11,43.046000,168.004506,0.041714


### KNN Tuning Results

K = 9 achieved the lowest MAE and RMSE and highest R^2 among the tested KNN values. Performance improved from K = 3 to 9, then declined slightly at K = 11.

# Model Selection

Both models are scored on the same 5,000 validation rows. Training sizes differ. RMSE guides selection because large errors pose inventory risks; MAE is also reported.

In [39]:
linear_sample_predictions = val_predictions[X_knn_val.index]

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "KNN (K=9)"],
    "MAE": [
        mean_absolute_error(y_knn_val, linear_sample_predictions),
        knn_results.loc[knn_results["K"] == 9, "MAE"].iloc[0]
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_knn_val, linear_sample_predictions)),
        knn_results.loc[knn_results["K"] == 9, "RMSE"].iloc[0]
    ],
    "R2": [
        r2_score(y_knn_val, linear_sample_predictions),
        knn_results.loc[knn_results["K"] == 9, "R2"].iloc[0]
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Linear Regression,43.186311,159.920170,0.131720
1,KNN (K=9),42.776089,167.792858,0.044127


### Model Selection Result

Linear Regression had lower RMSE and higher R^2 on the shared validation sample, while tuned KNN had slightly lower MAE. Linear Regression was selected for its lower RMSE.

## Training the Final Model

Linear Regression is retrained on the complete training data before final testing.

In [40]:
final_model = LinearRegression()

final_model.fit(X_train_ready, y_train)

print("Final model training completed.")

Final model training completed.


# Final Model Evaluation

The selected Linear Regression model is evaluated on the unseen test data.

In [41]:
test_predictions = final_model.predict(X_test_ready)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print("Final Test MAE:", test_mae)
print("Final Test RMSE:", test_rmse)
print("Final Test R²:", test_r2)

Final Test MAE: 41.88117784181777
Final Test RMSE: 163.1667548800459
Final Test R²: 0.09341827803816316


### Final Test Results

The model achieved an MAE of 41.88, RMSE of 163.17 and R² of 0.093. The higher RMSE indicates some large prediction errors, while the low R² shows that the model explains only a small proportion of demand variation.

# Model Strengths and Limitations

Linear Regression is fast, simple and interpretable, and had lower shared-sample validation RMSE than KNN, although KNN had lower MAE. Similar validation and test results also indicate consistent performance.

However, the low R² shows that the selected features explain little of the variation in demand. Broad product categories cannot distinguish individual products, while factors such as promotions, holidays and seasonality are unavailable. Real demand may also contain relationships that Linear Regression cannot capture.

# Business Implications and Recommendations

The model gives a rough demand estimate by city, category, and day—useful for early inventory planning.

But the low R² means don’t rely on it alone; big errors could cause overstock or stockouts.

To improve it, add richer product details, more sales history, and factors like promos, holidays, and seasonality; test stronger models if you need higher accuracy.

# Model Explainability

Coefficients describe associations conditional on other features, not causal effects. Magnitudes depend on units and encoding and are not directly comparable importance scores. Full one-hot encoding with an intercept makes individual coefficients non-unique.

In [42]:
coefficients = pd.DataFrame({
    "Feature": X_train_ready.columns,
    "Coefficient": final_model.coef_
})

coefficients["Absolute Coefficient"] = coefficients["Coefficient"].abs()

coefficients = coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)

coefficients.head(10)

,Feature,Coefficient,Absolute Coefficient
29,l0_category_Vegetables & Fruits,249.623431,249.623431
25,l0_category_Specials,192.504438,192.504438
14,l0_category_Dairy & Breakfast,63.725522,63.725522
9,l0_category_Baby Care,-45.695585,45.695585
19,l0_category_Organic & Premium,-43.218234,43.218234
16,l0_category_Home & Office,-41.223879,41.223879
20,l0_category_Paan Corner,-40.737858,40.737858
21,l0_category_Personal Care,-40.323113,40.323113
23,l0_category_Pharma & Wellness,-39.817108,39.817108
22,l0_category_Pet Care,-39.535698,39.535698


### Explainability Results

Under the fitted encoding, Vegetables & Fruits, Specials, and Dairy & Breakfast have positive coefficients; Baby Care and Organic & Premium have negative coefficients. These signs describe this parameterisation, not causal effects or comparisons with an omitted reference category.

# Deployment Considerations

The model could support inventory planning by generating demand estimates from new product, city and date data. The same preprocessing must be applied to new data and performance should be monitored over time. Due to its limited accuracy, the model should support rather than automate inventory decisions.

# Conclusion

Linear Regression was selected for lower validation RMSE, while tuned KNN had lower MAE. The selected model explained little demand variation. Richer product features and longer sales history are needed before operational use.

# Dataset Source

Historical e-commerce sales and product data were used to create daily product demand.

Dataset source: 
1. https://www.kaggle.com/datasets/iyumrahul/flipkartsalesdataset?select=Sales.csv 
2.  https://www.kaggle.com/datasets/iyumrahul/flipkartsalesdataset?select=products.csv

# References

Géron, A. (2022) *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd edn. Sebastopol, CA: O'Reilly Media.

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825–2830.

Dataset Reference

1. https://www.kaggle.com/datasets/iyumrahul/flipkartsalesdataset?select=Sales.csv 
2. https://www.kaggle.com/datasets/iyumrahul/flipkartsalesdataset?select=products.csv